In [1]:
import sys
import os

# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the parent directory to sys.path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [2]:
from helper import RAGHelper
import psycopg
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
# from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
import uuid
import re
from tenacity import retry, retry_if_exception_type, stop_after_attempt
from tenacity.wait import wait_base
load_dotenv()


True

In [3]:
class wait_for_groq_rate_limit(wait_base):
    def __init__(self, fallback_wait=60):
        self.fallback_wait = fallback_wait

    def __call__(self, retry_state):
        exception = retry_state.outcome.exception()
        if exception:
            error_msg = str(exception)
            
            # Regex to extract minutes and seconds from "try again in 13m19.199s"
            match = re.search(r'try again in (?:(\d+)m)?([\d.]+)s', error_msg)
            if match:
                minutes = int(match.group(1)) if match.group(1) else 0
                seconds = float(match.group(2))
                
                # Calculate total wait time in seconds + 2 seconds of buffer room
                wait_time = (minutes * 60) + seconds + 2.0 
                
                print(f"\n⏳ Groq Rate Limit! Pausing execution for {wait_time:.1f} seconds...")
                return wait_time
                
        # If it's a different error, fallback to 60 seconds
        return self.fallback_wait

In [4]:
@retry(
    retry=retry_if_exception_type(Exception), 
    wait=wait_for_groq_rate_limit(fallback_wait=60),
    stop=stop_after_attempt(5)
)
def process_single_question(q, session_id):
    # Extract data from the tuple
    qid = q[0]
    query = q[1]
    
    print(f"Processing QID {qid}...")
    
    # 1. Generate Response
    query, response, sent_list = rg.simple_rag(
        query=query, 
        expert_domain=expert_domain, 
        retriever=retriever,
        gen_model=gen_model
    )
    
    # 2. Evaluate
    eval_response = rg.evaluate_rag(query, response, sent_list, eval_model=eval_model)
    
    # 3. Calculate Metrics
    rg.get_metrics(eval_response, sent_list)
    
    # 4. Insert into Database
    rg.db_insert(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        table_name=eval_table, 
        database_url=DATABASE_URL,
        qid=qid, 
        vector_db=rag_db, 
        session_id=session_id,
        retrieval_type=retrieval_type
    )
    print(f"✅ Successfully inserted QID {qid}")

In [5]:
DATABASE_URL = os.getenv("DATABASE_URL")
questions_table = "nextgenrag_sample_questions" 
eval_table = "nextgenrag_v1"
retrieval_type = 'dense'
embedder = "BAAI/bge-base-en-v1.5" #BAAI/LLM-Embedder
search_type = "similarity"
search_kwargs = {"k":3}
embedding_fn = HuggingFaceEmbeddings(model=embedder)
chromadb_folder = "../database"

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
print(retrieval_type)

dense


In [7]:
rag_db = "Chroma"
db_name = "finance_1024_200"
collection_name = embedder.replace("/", "_")
persist_directory = f"{chromadb_folder}/{db_name}"
expert_domain = "finance"
chunk_size = int(db_name.split("_")[1])
chunk_overlap = int(db_name.split("_")[2])
gen_model = "llama-3.1-8b-instant"
eval_model = "openai/gpt-oss-20b"#"llama-3.3-70b-versatile"#"openai/gpt-oss-120b"

In [8]:

conn = psycopg.connect(DATABASE_URL)
cursor = conn.cursor()



In [9]:
sql_response = cursor.execute(f"""select a.id, a.query from {questions_table} as a left join {eval_table} as b 
on a.id = b.id and b.embed_model<> 'BAAI/LLM-Embedder'
where b.chunk_size is null limit 2""").fetchall()
conn.close()

In [10]:
# for r in sql_response:
#     print(r)

In [11]:
vector_db = Chroma(collection_name=collection_name, embedding_function=embedding_fn, persist_directory=persist_directory)
retriever = vector_db.as_retriever(search_type=search_type, search_kwargs=search_kwargs)

In [12]:
# eval_message = rg.eval_message

In [13]:
rg = RAGHelper(os.getenv("GROQ_API_KEY3"))

In [14]:
session_id = uuid.uuid4()

for q in sql_response:
#     process_single_question(q, session_id)
    
# print("🎉 All questions processed and evaluated!")
    
    query = q[1]
    query, response, sent_list = rg.simple_rag(query=query, expert_domain=expert_domain, retriever=retriever,gen_model=gen_model)
    eval_response = rg.evaluate_rag(query, response, sent_list, eval_model=eval_model)
    print(eval_response)
    relevance, utilization, completeness, adherence = rg.get_metrics(eval_response, sent_list)
    # rg.db_insert( chunk_size=chunk_size,
    #           chunk_overlap=chunk_overlap,
    #           table_name=eval_table, database_url=DATABASE_URL,
    #           qid=q[0], vector_db=rag_db, session_id=session_id,
    #               retrieval_type=retrieval_type)
    print(relevance, utilization, completeness, adherence)
    
    # print(f"Query: {query}\n response:{response}")

{"relevance_explanation":"The question asks for the largest cost of revenue item in Adjusted Revenue. The document contains a bullet point in sentence a0 that explicitly states \"Adjusted Revenue is net of transaction‑based costs, which is our largest cost of revenue item;\". This sentence directly answers the question. No other sentence is required to determine the answer, though sentence b1 lists the amount of transaction‑based costs, it does not state that it is the largest. Therefore, a0 is the key sentence for relevance.","all_relevant_sentence_keys":["a0"],"overall_supported_explanation":"The response makes a single claim: transaction‑based costs is the largest cost of revenue item. Sentence a0 explicitly makes that claim, so the response is fully supported by the source. No contradictory information is present in the documents. Thus the response is correct and fully supported.","overall_supported":true,"sentence_support_information":[{"response_sentence_key":"response1","explana

JSONDecodeError: Expecting value: line 1 column 1 (char 0)